In [1]:
import base64
import os
import requests
from bs4 import BeautifulSoup
from google import genai
from google.genai import types
from dotenv import load_dotenv


In [2]:
# Load environment variables
env_path = r"C:\Users\Cem\Desktop\projeler\YapayZekaUygulamalari\.env"
load_dotenv(env_path)


True

In [3]:
class GeminiWebScraper:
    def __init__(self, model_name="gemini-2.0-flash"):
        self.model_name = model_name
        self.client = genai.Client(
            api_key=os.getenv("GEMINI_API_KEY"),
        )
    
    def scrape_website(self, url):
        """Scrape website content and return title and text"""
        try:
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            
            title = soup.title.string if soup.title else "Title not found"
            
            # Remove unwanted elements
            for element in soup.find_all(['script', 'style', 'nav', 'header', 'footer', 'aside']):
                element.decompose()
            
            # Get main content
            main_content = soup.find('main') or soup.find('body')
            if main_content:
                text = main_content.get_text(separator="\n", strip=True)
            else:
                text = soup.get_text(separator="\n", strip=True)
            
            return {
                'title': title,
                'text': text,
                'url': url
            }
        except Exception as e:
            return f"Error scraping website: {str(e)}"
    
    def summarize_website(self, url, custom_prompt=None):
        """Summarize website content using Gemini"""
        # Scrape the website
        website_data = self.scrape_website(url)
        
        if isinstance(website_data, str):  # Error occurred
            return website_data
        
        # Default prompt for e-commerce analysis
        if custom_prompt is None:
            prompt = f"""Aşağıdaki e-ticaret sitesi içeriğini analiz et ve satılan ürünleri fiyatları ile listele:

SITE: {website_data['title']}

İÇERİK:
{website_data['text']} 

GÖREV:
- Sadece ürün kategorilerini ve türlerini listele
- Navigasyon menülerini görmezden gel
- Markdown formatında kısa özet ver
- Fiyat bilgisi varsa dahil et"""
        else:
            prompt = f"{custom_prompt}\n\nSITE: {website_data['title']}\n\nİÇERİK:\n{website_data['text']}"
        
        # Create content for Gemini
        contents = [
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text=prompt),
                ],
            ),
        ]
        
        # Generate content
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=contents,
            )
            return response.text
        except Exception as e:
            return f"Error generating summary: {str(e)}"


In [4]:
# Create an instance of the Gemini Web Scraper
scraper = GeminiWebScraper()


In [5]:
# Simple function to summarize any website (equivalent to the original simple_summarize)
def simple_summarize(url):
    """Simple function to summarize a website using Gemini"""
    return scraper.summarize_website(url)


In [6]:
# Test the implementation with Trendyol website
result = simple_summarize("https://www.trendyol.com")
print(result)


İşte e-ticaret sitesindeki ürünlerin ve fiyatlarının listesi:

*   **Kerastase Genesis Sérum Anti-Chute Fortifiant Önleyici Serum 90 ml:** 2.849,00 TL (İndirimli fiyatı 3.049,00 TL)
*   **REDZARRAM etek yan cepli efekli soluk kahve mini etek:** 649,00 TL
*   **SİNSER TEKS Kapşonlu Çocuk KUROMI Baskılı Eşofman Takımı:** 279,99 TL
*   **HELİS BUTİK Fırfır Detay Bordo Elbise:** 2.275,00 TL
*   **Bershka Örgü detaylı polo yaka hırka:** 1.590,00 TL
*   **Morfose Milk Therapy Saç Maskesi 500 ml:** 234,90 TL
*   **Gülşah Evecen Özel Tasarım Siyah Simli Büzgü Paça Cargo Eşofman:** 1.200,00 TL
*   **DeFacto Kız Bebek 1,6 Tog Elyaf Desenli Fermuarlı Uzun Kollu Tulum F3565A525WN:** 1.631,99 TL
*   **MACERATİ Swank Unisex Siyah Kare Çerçeve Blue Block Mavi Işık Filtreli Ekran Gözlüğü:** 177,15 TL (Trendyol Plus Özel fiyatı, normal fiyat 186,47 TL)
*   **renklidüşler Sarı& Limon Kokulu Salkım Üzüm Sabun & Musluk Sabunu Elyapımı:** 214,94 TL
*   **Tala Tüy Dökücü Tüy Azaltıcı Karınca Yumurtası Yağı 

In [ ]:
# Example with custom prompt
custom_prompt = """Bu web sitesini analiz et ve şu bilgileri ver:
1. Site türü (e-ticaret, haber, blog vb.)
2. Ana ürün/hizmet kategorileri
3. Hedef kitle
4. Öne çıkan özellikler

Markdown formatında düzenli bir rapor hazırla."""

custom_result = scraper.summarize_website("https://www.trendyol.com", custom_prompt)
print(custom_result)


In [ ]:
# Display results in markdown format (optional)
from IPython.display import Markdown, display

def display_summary(url, custom_prompt=None):
    """Display website summary in markdown format"""
    if custom_prompt:
        summary = scraper.summarize_website(url, custom_prompt)
    else:
        summary = simple_summarize(url)
    display(Markdown(summary))

# Uncomment the line below to display the summary in markdown format
# display_summary("https://www.trendyol.com")
